# 06 — RQ3 socioeconomic and healthcare-access differences

### Research question

**Among adults with HICP, does reported use of pain-management strategies vary across socioeconomic and healthcare-access characteristics?**

This is mainly an *exploratory comparison*. I want to see whether reported strategy use looks different across:

- income / poverty ratio
- education
- region
- delayed medical care because of cost
- did not get medical care because of cost

I am not starting with a directional hypothesis here. Before comparing groups, I first check how these variables are coded and how much usable data I have for each one.

## Setup

I recreate the same HICP analytical group used in RQ2 and then inspect the socioeconomic and healthcare-access variables needed for RQ3.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt

from src.hicp import flag_chronic_pain, flag_hicp, flag_cohort

df = pd.read_csv("../data/raw/adult25csv/adult25.csv", low_memory=False)

df["chronic_pain"] = flag_chronic_pain(df)
df["hicp"] = flag_hicp(df)

cohort = flag_cohort(df)

analysis_df = df.loc[cohort].copy()
hicp_df = analysis_df.loc[analysis_df["hicp"]].copy()

hicp_df.shape

(1956, 579)

In [2]:
rq3_vars = {
    "POVRATTC_A": "Income / poverty ratio",
    "EDUCP_A": "Education",
    "REGION": "Region",
    "MEDDL12M_A": "Delayed medical care due to cost during the past 12 months",
    "MEDNG12M_A": "Did not get medical care due to cost, past 12 months"
}

for var, label in rq3_vars.items():
    print(f"\n{label} ({var})")
    print(hicp_df[var].value_counts(dropna=False).sort_index())


Income / poverty ratio (POVRATTC_A)
POVRATTC_A
0.00     36
0.04      1
0.05      2
0.06      1
0.07      1
         ..
11.64     1
11.88     1
11.89     1
11.90     4
12.00    40
Name: count, Length: 493, dtype: int64

Education (EDUCP_A)
EDUCP_A
1     200
2      36
3      91
4     480
5     340
6      98
7     223
8     289
9     149
10     45
97      3
99      2
Name: count, dtype: int64

Region (REGION)
REGION
1    347
2    423
3    737
4    449
Name: count, dtype: int64

Delayed medical care due to cost during the past 12 months (MEDDL12M_A)
MEDDL12M_A
1     295
2    1661
Name: count, dtype: int64

Did not get medical care due to cost, past 12 months (MEDNG12M_A)
MEDNG12M_A
1     281
2    1675
Name: count, dtype: int64


## Preparing the RQ3 groups

-> Some of the RQ3 variables can be used almost as they are, while **income and education need broader groups** before comparing strategy use.

-> For income, POVRATTC_A is a continuous family poverty ratio and is top-coded at 12, so I group it into broader ranges instead of comparing hundreds of individual values.

-> Education also has 10 different levels, so I combine similar levels into broader categories. 

-> Region and the two cost-related access variables can be labelled directly from the NHIS codebook.

##### Income:

In [3]:
hicp_df["income_group"] = pd.cut(
    hicp_df["POVRATTC_A"],
    bins=[-0.01, 1, 2, 4, float("inf")],
    labels=[
        "< 1.0", # lowest income/poverty-ratio group
        "1.0 to < 2.0",
        "2.0 to < 4.0",
        "4.0+"  # highest group
    ],
    right=False
)

##### Education:

In [4]:
education_map = {
    1: "Less than high school",
    2: "Less than high school",
    3: "Less than high school",
    4: "High school graduate",
    5: "Some college / associate",
    6: "Some college / associate",
    7: "Some college / associate",
    8: "Bachelor's or higher",
    9: "Bachelor's or higher",
    10: "Bachelor's or higher"
}

hicp_df["education_group"] = hicp_df["EDUCP_A"].map(education_map)

##### Region:

In [5]:
region_map = {
    1: "Northeast",
    2: "Midwest",
    3: "South",
    4: "West"
}

hicp_df["region_group"] = hicp_df["REGION"].map(region_map)

##### Healthcare access:

In [6]:
yes_no_map = {1: "Yes", 2: "No"}

hicp_df["delayed_care_cost"] = hicp_df["MEDDL12M_A"].map(yes_no_map)
hicp_df["no_care_cost"] = hicp_df["MEDNG12M_A"].map(yes_no_map)

In [7]:
for var in ["income_group", "education_group", "region_group", "delayed_care_cost", "no_care_cost"]:
    print(f"\n{var}")
    print(hicp_df[var].value_counts(dropna=False))


income_group
income_group
2.0 to < 4.0    553
1.0 to < 2.0    546
4.0+            480
< 1.0           377
Name: count, dtype: int64

education_group
education_group
Some college / associate    661
Bachelor's or higher        483
High school graduate        480
Less than high school       327
NaN                           5
Name: count, dtype: int64

region_group
region_group
South        737
West         449
Midwest      423
Northeast    347
Name: count, dtype: int64

delayed_care_cost
delayed_care_cost
No     1661
Yes     295
Name: count, dtype: int64

no_care_cost
no_care_cost
No     1675
Yes     281
Name: count, dtype: int64


### Group sizes

-> For the deeper comparison, I use the criteria that I had already defined before looking at the results: no subgroup below 50 respondents, and enough variation across the socioeconomic/access groups to be worth exploring further.

-> All of the groups are large enough for the planned comparisons. The smallest subgroup is still well above 50 respondents.

**Reminder for myself:** 50 is a practical guardrail to avoid focusing on percentages from very small groups, not a formal power threshold.

-> Education has 5 unclassified responses because the special NHIS codes (97 and 99) were not mapped to an education level. These respondents will only be excluded from analyses involving education.

## Comparing pain-management strategy use

-> Now that the socioeconomic and healthcare-access groups are defined and the group sizes look usable, I can compare reported strategy use across them.

In [8]:
pain_management_vars = {
    "PAIOTCMEDS_A": "Over-the-counter medication",
    "PAIOPIOID_A": "Opioids",
    "PAIPRSMEDS_A": "Prescribed pain reliever",
    "PAIPHYSTPY_A": "Physical therapy",
    "PAICHIRO_A": "Chiropractic care",
    "PAITALKTPY_A": "Talk therapy",
    "PAIYOGA_A": "Yoga",
    "PAIEXRCISE_A": "Exercise",
    "PAIMASSAGE_A": "Massage",
    "PAIMEDITAT_A": "Meditation",
    "PAIMOTHER_A": "Other approaches"
}

### Income / poverty ratio

I start by comparing the 11 pain-management strategies across the four income / poverty groups.

As in RQ2, **the percentage for each strategy is calculated using only valid Yes/No responses.**

In [9]:
income_results = []

for income in hicp_df["income_group"].cat.categories:
    income_df = hicp_df[hicp_df["income_group"] == income]

    for var, label in pain_management_vars.items():
        responses = income_df[var]
        valid_responses = responses[responses.isin([1, 2])]

        yes_n = (valid_responses == 1).sum()
        valid_n = len(valid_responses)
        percent_yes = (yes_n / valid_n) * 100

        income_results.append({
            "Income group": income,
            "Strategy": label,
            "Valid n": valid_n,
            "Percent": percent_yes
        })

In [10]:
income_results_df = pd.DataFrame(income_results)

In [11]:
income_results_df["Percent"] = income_results_df["Percent"].round(1)

income_results_df.head(10)

,Income group,Strategy,Valid n,Percent
0,< 1.0,Over-the-counter medication,376,72.9
1,< 1.0,Opioids,374,33.2
2,< 1.0,Prescribed pain reliever,371,33.7
3,< 1.0,Physical therapy,374,30.7
4,< 1.0,Chiropractic care,374,10.2
5,< 1.0,Talk therapy,373,7.8
6,< 1.0,Yoga,374,6.4
7,< 1.0,Exercise,374,47.9
8,< 1.0,Massage,374,13.9
9,< 1.0,Meditation,374,21.4


In [12]:
income_table = income_results_df.pivot(index="Strategy", columns="Income group", values="Percent")

income_table

Income group,1.0 to < 2.0,2.0 to < 4.0,4.0+,< 1.0
Strategy,,,,
Chiropractic care,9.7,11.6,12.5,10.2
Exercise,48.1,55.0,62.3,47.9
Massage,15.6,18.7,23.5,13.9
Meditation,17.5,21.6,20.5,21.4
Opioids,33.9,33.3,31.3,33.2
Other approaches,20.6,24.5,27.1,22.7
Over-the-counter medication,77.5,78.6,80.6,72.9
Physical therapy,26.2,33.8,36.9,30.7
Prescribed pain reliever,32.5,35.0,34.0,33.7


In [13]:
income_order = ["< 1.0", "1.0 to < 2.0","2.0 to < 4.0", "4.0+"]

income_table = income_table[income_order]

income_table

Income group,< 1.0,1.0 to < 2.0,2.0 to < 4.0,4.0+
Strategy,,,,
Chiropractic care,10.2,9.7,11.6,12.5
Exercise,47.9,48.1,55.0,62.3
Massage,13.9,15.6,18.7,23.5
Meditation,21.4,17.5,21.6,20.5
Opioids,33.2,33.9,33.3,31.3
Other approaches,22.7,20.6,24.5,27.1
Over-the-counter medication,72.9,77.5,78.6,80.6
Physical therapy,30.7,26.2,33.8,36.9
Prescribed pain reliever,33.7,32.5,35.0,34.0


In [14]:
income_variation = income_results_df.groupby("Strategy")["Percent"].agg(Min="min",Max="max")

In [15]:
income_variation["Difference"] = (income_variation["Max"] - income_variation["Min"]).round(1)

In [16]:
income_variation = income_variation.sort_values("Difference",ascending=False)

In [17]:
income_variation

,Min,Max,Difference
Strategy,,,
Exercise,47.9,62.3,14.4
Physical therapy,26.2,36.9,10.7
Massage,13.9,23.5,9.6
Over-the-counter medication,72.9,80.6,7.7
Other approaches,20.6,27.1,6.5
Meditation,17.5,21.6,4.1
Yoga,6.4,10.4,4.0
Chiropractic care,9.7,12.5,2.8
Opioids,31.3,33.9,2.6


### First look at income differences

-> **Exercise shows the largest difference** across income groups (14.4 percentage points), followed by **physical therapy** (10.7) and **massage** (9.6).

-> The 'Difference' column only shows the gap between the highest and lowest percentage -> It does not tell me whether there is a clear trend across the income groups.

-> Exercise and massage increase fairly consistently as income rises, while physical therapy varies more irregularly.

-> **Opioids, prescribed pain relievers and talk therapy change very little** across income groups.

### Education

Next I compare reported strategy use across the four education groups.

The 5 respondents with unclassified education responses are not included in this comparison. As before, each percentage uses only valid Yes/No responses for that strategy.

In [18]:
education_order = [
    "Less than high school",
    "High school graduate",
    "Some college / associate",
    "Bachelor's or higher"
]

In [19]:
education_results = []

for education in education_order:
    education_df = hicp_df[hicp_df["education_group"] == education]

    for var, label in pain_management_vars.items():
        responses = education_df[var]
        valid_responses = responses[responses.isin([1, 2])]

        yes_n = (valid_responses == 1).sum()
        valid_n = len(valid_responses)
        percent_yes = (yes_n / valid_n) * 100

        education_results.append({
            "Education group": education,
            "Strategy": label,
            "Valid n": valid_n,
            "Percent": percent_yes
        })

In [20]:
education_results_df = pd.DataFrame(education_results)

education_results_df["Percent"] = education_results_df["Percent"].round(1)

In [21]:
education_table = education_results_df.pivot(index="Strategy", columns="Education group", values="Percent")

education_table = education_table[education_order]

In [22]:
education_table

Education group,Less than high school,High school graduate,Some college / associate,Bachelor's or higher
Strategy,,,,
Chiropractic care,7.7,7.7,12.9,14.1
Exercise,39.3,50.2,55.0,64.4
Massage,12.6,13.6,19.4,24.1
Meditation,10.8,12.8,23.3,29.4
Opioids,31.3,32.1,35.3,31.4
Other approaches,19.9,19.0,24.4,29.7
Over-the-counter medication,75.5,75.7,79.0,79.0
Physical therapy,21.5,25.9,33.3,42.6
Prescribed pain reliever,32.9,32.0,33.9,36.1


### First look at education differences

-> **The education differences are more pronounced than the income differences.**

-> Several **non-pharmacological** strategies - *exercise, physical therapy, meditation, massage and yoga* - **increase fairly consistently across education levels**. 

-> In contrast, over-the-counter medication, opioids and prescribed pain relievers vary much less across education groups.

-> At this stage these are descriptive patterns only. They do not show that education itself causes differences in strategy use.

In [23]:
education_variation = education_results_df.groupby("Strategy")["Percent"].agg(Min="min", Max="max")

In [24]:
education_variation["Difference"] = (education_variation["Max"] - education_variation["Min"]).round(1)

In [25]:
education_variation = education_variation.sort_values("Difference", ascending=False)

In [26]:
education_variation

,Min,Max,Difference
Strategy,,,
Exercise,39.3,64.4,25.1
Physical therapy,21.5,42.6,21.1
Meditation,10.8,29.4,18.6
Massage,12.6,24.1,11.5
Other approaches,19.0,29.7,10.7
Yoga,2.8,13.1,10.3
Talk therapy,2.9,11.2,8.3
Chiropractic care,7.7,14.1,6.4
Prescribed pain reliever,32.0,36.1,4.1


-> **Exercise, physical therapy and meditation** currently show the **strongest variation across education groups**

> Obvioously, they are the main candidates to keep an eye on when deciding whether one strategy deserves a deeper analysis.

### Region

Next I compare reported strategy use across the four US regions.

As before, each percentage is calculated using only valid Yes/No responses for that strategy.

In [27]:
region_order = ["Northeast", "Midwest", "South", "West"]

In [28]:
region_results = []

for region in region_order:
    region_df = hicp_df[hicp_df["region_group"] == region]

    for var, label in pain_management_vars.items():
        responses = region_df[var]
        valid_responses = responses[responses.isin([1, 2])]

        yes_n = (valid_responses == 1).sum()
        valid_n = len(valid_responses)
        percent_yes = (yes_n / valid_n) * 100

        region_results.append({
            "Region": region,
            "Strategy": label,
            "Valid n": valid_n,
            "Percent": percent_yes
        })

In [29]:
region_results_df = pd.DataFrame(region_results)
region_results_df["Percent"] = region_results_df["Percent"].round(1)

In [30]:
region_table = region_results_df.pivot(index="Strategy", columns="Region", values="Percent")
region_table = region_table[region_order]

In [31]:
region_table

Region,Northeast,Midwest,South,West
Strategy,,,,
Chiropractic care,11.6,14.3,9.1,10.7
Exercise,49.3,52.9,50.4,62.4
Massage,14.5,18.3,16.2,23.8
Meditation,19.4,16.9,18.4,26.6
Opioids,25.7,35.9,35.2,32.1
Other approaches,23.8,23.6,19.8,30.1
Over-the-counter medication,78.3,76.3,78.1,77.7
Physical therapy,33.0,30.4,28.1,38.3
Prescribed pain reliever,33.1,35.2,34.9,31.2


### First look at regional differences

-> Regional differences are less structured than the patterns seen for education, since region has no natural order.

-> **Exercise is notably higher in the West**, and physical therapy, massage, meditation, yoga and "other approaches" are also reported more often there than in some of the other regions.

-> Opioid use is lowest in the Northeast, while over-the-counter medication is very similar across all four regions.

-> These are descriptive regional differences only and do not explain why the regions differ.

In [32]:
region_variation = region_results_df.groupby("Strategy")["Percent"].agg(Min="min", Max="max")

In [33]:
region_variation["Difference"] = (region_variation["Max"] - region_variation["Min"]).round(1)

In [34]:
region_variation = region_variation.sort_values("Difference", ascending=False)

In [35]:
region_variation

,Min,Max,Difference
Strategy,,,
Exercise,49.3,62.4,13.1
Other approaches,19.8,30.1,10.3
Opioids,25.7,35.9,10.2
Physical therapy,28.1,38.3,10.2
Meditation,16.9,26.6,9.7
Massage,14.5,23.8,9.3
Chiropractic care,9.1,14.3,5.2
Talk therapy,5.2,10.4,5.2
Yoga,6.7,11.6,4.9


*Disclaimer*: I do not try to explain these regional differences here, as region may capture many different factors that are not measured or separated in this analysis.

### Delayed medical care because of cost

I now compare strategy use between people who did and did not report delaying medical care because of cost in the past 12 months.

This variable has two groups (Yes / No), and each strategy percentage is again based only on valid Yes/No responses.

In [36]:
delayed_care_results = []

for access_group in ["Yes", "No"]:
    access_df = hicp_df[
        hicp_df["delayed_care_cost"] == access_group
    ]

    for var, label in pain_management_vars.items():
        responses = access_df[var]
        valid_responses = responses[responses.isin([1, 2])]

        yes_n = (valid_responses == 1).sum()
        valid_n = len(valid_responses)
        percent_yes = (yes_n / valid_n) * 100

        delayed_care_results.append({
            "Delayed care due to cost": access_group,
            "Strategy": label,
            "Valid n": valid_n,
            "Percent": percent_yes
        })

In [37]:
delayed_care_df = pd.DataFrame(delayed_care_results)

delayed_care_df["Percent"] = delayed_care_df["Percent"].round(1)

In [38]:
delayed_care_table = delayed_care_df.pivot(index="Strategy", columns="Delayed care due to cost", values="Percent")
delayed_care_table = delayed_care_table[["No", "Yes"]]

In [39]:
delayed_care_table

Delayed care due to cost,No,Yes
Strategy,,
Chiropractic care,10.6,13.2
Exercise,52.3,60.2
Massage,17.6,21.1
Meditation,18.5,29.0
Opioids,33.6,29.6
Other approaches,23.2,26.2
Over-the-counter medication,76.4,85.1
Physical therapy,32.4,28.5
Prescribed pain reliever,34.2,31.5


### First look at delayed care due to cost

-> There are some noticeable differences between people who did and did not report delaying medical care because of cost.

-> People reporting this cost barrier have higher reported use of several strategies, including over-the-counter medication, exercise, meditation, talk therapy and yoga.

-> Physical therapy, opioids and prescribed pain relievers are slightly lower in the group that reported delayed care.

> The pattern is not simple enough to interpret as people with cost barriers only shifting towards cheaper or more accessible strategies, **since some of the strategies reported more often may also involve cost**.

-> These are descriptive differences only. I cannot tell from this analysis why these patterns occur or whether the cost barrier itself explains them.

In [40]:
delayed_care_difference = delayed_care_table.copy()

delayed_care_difference["Difference"] = (delayed_care_difference["Yes"] - delayed_care_difference["No"]).round(1)

In [41]:
delayed_care_difference["Absolute difference"] = (delayed_care_difference["Difference"].abs())

In [42]:
delayed_care_difference = delayed_care_difference.sort_values("Absolute difference", ascending=False)

In [43]:
delayed_care_difference

Delayed care due to cost,No,Yes,Difference,Absolute difference
Strategy,,,,
Meditation,18.5,29.0,10.5,10.5
Over-the-counter medication,76.4,85.1,8.7,8.7
Talk therapy,5.6,14.3,8.7,8.7
Exercise,52.3,60.2,7.9,7.9
Yoga,7.3,12.6,5.3,5.3
Opioids,33.6,29.6,-4.0,4.0
Physical therapy,32.4,28.5,-3.9,3.9
Massage,17.6,21.1,3.5,3.5
Other approaches,23.2,26.2,3.0,3.0


### Did not get medical care because of cost

Finally, I compare strategy use between people who did and did not report needing medical care but not getting it because of cost in the past 12 months.

Again, each percentage is based only on valid Yes/No responses for that strategy.

In [44]:
no_care_results = []

for access_group in ["Yes", "No"]:
    access_df = hicp_df[
        hicp_df["no_care_cost"] == access_group
    ]

    for var, label in pain_management_vars.items():
        responses = access_df[var]
        valid_responses = responses[responses.isin([1, 2])]

        yes_n = (valid_responses == 1).sum()
        valid_n = len(valid_responses)
        percent_yes = (yes_n / valid_n) * 100

        no_care_results.append({
            "Did not get care due to cost": access_group,
            "Strategy": label,
            "Valid n": valid_n,
            "Percent": percent_yes
        })

In [45]:
no_care_df = pd.DataFrame(no_care_results)

no_care_df["Percent"] = no_care_df["Percent"].round(1)

In [46]:
no_care_table = no_care_df.pivot(index="Strategy", columns="Did not get care due to cost", values="Percent")
no_care_table = no_care_table[["No", "Yes"]]

In [47]:
no_care_table

Did not get care due to cost,No,Yes
Strategy,,
Chiropractic care,10.7,13.2
Exercise,52.6,58.6
Massage,17.6,21.4
Meditation,18.6,29.0
Opioids,33.6,29.3
Other approaches,22.9,28.2
Over-the-counter medication,76.9,82.6
Physical therapy,32.2,29.5
Prescribed pain reliever,34.0,32.4


### First look at not getting care due to cost

-> **A similar pattern appears for people who reported needing medical care but not getting it because of cost**.

-> Meditation and talk therapy show the largest differences, and exercise, yoga, over-the-counter medication and "other approaches" are also reported more often in the group that experienced this cost barrier.

-> Opioids and physical therapy are slightly lower in this group.

-> Again, the pattern is mixed rather than a simple shift towards cheaper strategies, so I keep the interpretation descriptive rather than trying to explain why these differences occur.

In [48]:
no_care_difference = no_care_table.copy()

no_care_difference["Difference"] = (no_care_difference["Yes"] - no_care_difference["No"]).round(1)

In [49]:
no_care_difference["Absolute difference"] = (no_care_difference["Difference"].abs())

In [50]:
no_care_difference = no_care_difference.sort_values( "Absolute difference", ascending=False)

In [51]:
no_care_difference

Did not get care due to cost,No,Yes,Difference,Absolute difference
Strategy,,,,
Meditation,18.6,29.0,10.4,10.4
Talk therapy,5.6,14.6,9.0,9.0
Exercise,52.6,58.6,6.0,6.0
Yoga,7.2,13.2,6.0,6.0
Over-the-counter medication,76.9,82.6,5.7,5.7
Other approaches,22.9,28.2,5.3,5.3
Opioids,33.6,29.3,-4.3,4.3
Massage,17.6,21.4,3.8,3.8
Physical therapy,32.2,29.5,-2.7,2.7


## Comparing variation across all RQ3 characteristics

-> I have now compared all 11 strategies across income, education, region and the two cost-related access barriers.

-> To decide whether one strategy is worth looking at more closely, I bring the differences together in one table.

-> For income, education and region, the value is the gap between the highest and lowest group. For the two cost barriers, it is the absolute difference between Yes and No.

In [52]:
rq3_summary = pd.DataFrame({
    "Income": income_variation["Difference"],
    "Education": education_variation["Difference"],
    "Region": region_variation["Difference"],
    "Delayed care due to cost": delayed_care_difference["Absolute difference"],
    "Did not get care due to cost": no_care_difference["Absolute difference"]
})

rq3_summary

,Income,Education,Region,Delayed care due to cost,Did not get care due to cost
Strategy,,,,,
Chiropractic care,2.8,6.4,5.2,2.6,2.5
Exercise,14.4,25.1,13.1,7.9,6.0
Massage,9.6,11.5,9.3,3.5,3.8
Meditation,4.1,18.6,9.7,10.5,10.4
Opioids,2.6,4.0,10.2,4.0,4.3
Other approaches,6.5,10.7,10.3,3.0,5.3
Over-the-counter medication,7.7,3.5,2.0,8.7,5.7
Physical therapy,10.7,21.1,10.2,3.9,2.7
Prescribed pain reliever,2.5,4.1,4.0,2.7,1.6


In [53]:
rq3_summary["Average difference"] = (rq3_summary.mean(axis=1)).round(1)

In [54]:
rq3_summary = rq3_summary.sort_values("Average difference", ascending=False)

In [55]:
rq3_summary

,Income,Education,Region,Delayed care due to cost,Did not get care due to cost,Average difference
Strategy,,,,,,
Exercise,14.4,25.1,13.1,7.9,6.0,13.3
Meditation,4.1,18.6,9.7,10.5,10.4,10.7
Physical therapy,10.7,21.1,10.2,3.9,2.7,9.7
Massage,9.6,11.5,9.3,3.5,3.8,7.5
Other approaches,6.5,10.7,10.3,3.0,5.3,7.2
Talk therapy,1.4,8.3,5.2,8.7,9.0,6.5
Yoga,4.0,10.3,4.9,5.3,6.0,6.1
Over-the-counter medication,7.7,3.5,2.0,8.7,5.7,5.5
Opioids,2.6,4.0,10.2,4.0,4.3,5.0


-> The average difference is only a summary of the five descriptive gaps

-> It is NOT a statistical score and does not automatically determine which strategy should be selected for deeper analysis.

### Selecting one strategy for a closer look

-> Based on the criteria defined before looking at the results, I select **exercise** for the deeper analysis.

> **All of the socioeconomic and access subgroups are large enough, and exercise shows relatively large differences across several of the RQ3 characteristics rather than only one of them.**

-> It also has the largest average descriptive gap across the five comparisons (13.3 percentage points)

-> The choice is based on the overall pattern: **exercise shows substantial variation across income, education and region, with smaller differences also present for the two cost-related access measures.**

----

## Looking more closely at exercise

**Exercise** was the strategy that stood out the most across the different RQ3 comparisons.

It met the two criteria I had defined before looking at the results:
- the subgroup sizes were large enough
- the differences were not limited to just one socioeconomic or access characteristic.

I now look at the exercise results together to make the pattern easier to see.|

#### Income

In [56]:
exercise_income = income_results_df[income_results_df["Strategy"] == "Exercise"][["Income group", "Valid n", "Percent"]]
exercise_income = exercise_income.reset_index(drop=True)

In [57]:
exercise_income

,Income group,Valid n,Percent
0,< 1.0,374,47.9
1,1.0 to < 2.0,545,48.1
2,2.0 to < 4.0,551,55.0
3,4.0+,480,62.3


#### Education

In [58]:
exercise_education = education_results_df[education_results_df["Strategy"] == "Exercise"][["Education group", "Valid n", "Percent"]]
exercise_education = exercise_education.reset_index(drop=True)

In [59]:
exercise_education

,Education group,Valid n,Percent
0,Less than high school,326,39.3
1,High school graduate,478,50.2
2,Some college / associate,660,55.0
3,Bachelor's or higher,481,64.4


#### Region

In [60]:
exercise_region = region_results_df[region_results_df["Strategy"] == "Exercise"][["Region", "Valid n", "Percent"]]
exercise_region = exercise_region.reset_index(drop=True)

In [61]:
exercise_region

,Region,Valid n,Percent
0,Northeast,345,49.3
1,Midwest,420,52.9
2,South,736,50.4
3,West,449,62.4


#### Delayed care due to cost

In [62]:
exercise_delayed = delayed_care_df[delayed_care_df["Strategy"] == "Exercise"][["Delayed care due to cost", "Valid n", "Percent"]]
exercise_delayed = exercise_delayed.reset_index(drop=True)

In [63]:
exercise_delayed

,Delayed care due to cost,Valid n,Percent
0,Yes,294,60.2
1,No,1656,52.3


#### Did not get care due to cost

In [64]:
exercise_no_care = no_care_df[no_care_df["Strategy"] == "Exercise"][["Did not get care due to cost", "Valid n", "Percent"]]
exercise_no_care = exercise_no_care.reset_index(drop=True)

In [65]:
exercise_no_care

,Did not get care due to cost,Valid n,Percent
0,Yes,280,58.6
1,No,1670,52.6


### What stands out for exercise?

-> **The clearest differences are in income and education**:
- Exercise increases across the income groups, from 47.9% in the lowest group to 62.3% in the highest
- The education pattern is even clearer: it goes from 39.3% among people with less than high school education to 64.4% among people with a bachelor's degree or higher.

-> Region is less easy to interpret, but the West has the highest reported exercise use (62.4%).

-> Exercise is also reported a bit more often among people who experienced either of the two cost-related barriers to medical care.

These are still descriptive patterns. I can see that exercise use varies across these groups, but I cannot say from this analysis why those differences exist.

----

## RQ3 — Final interpretation:

1. **Reported pain-management strategy** use does **vary across socioeconomic and healthcare-access characteristics** in this HICP sample.

2. The **clearest differences appeared across education**. Several **non-pharmacological strategies**, especially *exercise*, *physical therapy* and *meditation*, were reported more often **at higher education levels**.

3. **Income** also showed differences, **particularly for exercise and massage**, while regional patterns were more mixed.

4. The two cost-related access measures also showed differences in strategy use, but not in a simple "less access = less treatment" direction.

5. **Exercise stood out across several comparisons and was therefore selected for a closer look.** Its reported use increased across both income and education groups, was highest in the West, and was also somewhat higher among people reporting the two cost-related barriers.

These results are descriptive and do not show that socioeconomic or access characteristics cause the differences in strategy use.